## 1. Carga y exploración inicial del dataset


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

import warnings
import os
import gc
import time
import pickle
from tqdm import tqdm
from copy import deepcopy

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

# ── Reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ── Dispositivo
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch:  {torch.__version__}')
print(f'Pandas:   {pd.__version__}')
print(f'NumPy:    {np.__version__}')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(1)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch:  2.11.0+cu126
Pandas:   3.0.2
NumPy:    2.4.3
Dispositivo: cuda
GPU: NVIDIA RTX A6000
VRAM: 50.9 GB


In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(1))

Torch version: 2.11.0+cu126
CUDA disponible: True
GPU: NVIDIA RTX A6000


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("trafico-csv")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 18:35:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
DATA_PATH = "../data/Trafico_MODELOS2.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("encoding", "utf-8")
    .option("sep", ",")
    .csv(DATA_PATH)
)

df.show(5, truncate=False)
df.printSchema()

+----+---------+------------------+---------+-----+----+----------------------------------------+-----------------+-----------------+----------+-------------------------------------+----------------+----------+-----------+------+-----------------+-------------------+-------------+----------------------+-------------+--------------+-----------------+
|id  |tipo_elem|intensidad_trafico|ocupacion|carga|vmed|nombre                                  |longitud         |latitud          |Dia_semana|laborable / festivo / domingo festivo|Tipo de Festivo |Festividad|precip_mm  |llueve|intensidad_lluvia|datetime           |hay_accidente|tiempo_desde_accidente|heridos_leves|heridos_graves|victimas_mortales|
+----+---------+------------------+---------+-----+----+----------------------------------------+-----------------+-----------------+----------+-------------------------------------+----------------+----------+-----------+------+-----------------+-------------------+-------------+---------------